# Notebook 04 — Threshold Engineering & Deep Diagnostics

After training the VAE, reconstruction error is a *continuous* anomaly score.
This notebook finds the best **decision threshold** and explains *why* the model works.

| Strategy | Optimises | When to use |
|----------|-----------|-------------|
| **Max F1** | Harmonic mean of precision & recall | General-purpose balanced decision |
| **Min Cost** | Total monetary cost (cost matrix) | Business-driven trade-off |
| **Recall ≥ 90%** | Catches ≥90% of all fraud | Regulatory / compliance mandate |

**Sections 9–12** add diagnostic plots:
- Feature-wise reconstruction error (which features expose fraud)
- Anomaly score vs Amount (bias check)
- Latent space T-SNE (encoder quality)
- Visual confusion matrix (business communication)

## 0. Colab Setup

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Install / upgrade packages not pre-installed on Colab ────────────────────
import subprocess, sys
pkgs = ["optuna"]   # torch, sklearn, matplotlib, numpy already on Colab
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

print("Drive mounted and packages ready.")

In [ ]:
from pathlib import Path

# ── Project root on Drive ─────────────────────────────────────────────────────
DRIVE_ROOT = Path("/content/drive/MyDrive/Fraud_VAE_Project")

# ── Add project root to Python path so `src` package is importable ────────────
if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

# ── Key paths (override local cfg values) ─────────────────────────────────────
PROC_DIR   = DRIVE_ROOT / "data" / "processed"   # npy arrays live here
CKPT_PATH  = DRIVE_ROOT / "model" / "best_model.pth"
OUTPUT_DIR = DRIVE_ROOT / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check
print(f"Processed data : {PROC_DIR}  exists={PROC_DIR.exists()}")
print(f"Checkpoint     : {CKPT_PATH}  exists={CKPT_PATH.exists()}")
print(f"Output dir     : {OUTPUT_DIR}")
missing = [p for p in [
    PROC_DIR / "X_val.npy", PROC_DIR / "y_val.npy",
    PROC_DIR / "X_test.npy", PROC_DIR / "y_test.npy",
] if not p.exists()]
if missing:
    print("\nMISSING FILES:")
    for p in missing: print(f"  {p}")
else:
    print("\nAll required files found.")

## 1. Setup

In [ ]:
import sys, json
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.evaluate import (
    compute_anomaly_scores,
    compute_per_feature_errors,
    compute_latent_mu,
    plot_feature_reconstruction_error,
    plot_latent_tsne,
    plot_score_vs_amount,
    plot_confusion_matrix_heatmap,
    threshold_analysis,
    find_optimal_thresholds,
    plot_threshold_curves,
    evaluate_at_threshold,
)
from src.train import _build_feature_weights
import src.config as cfg

FEATURE_COLS = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
AMOUNT_IDX   = FEATURE_COLS.index("Amount")   # = 29

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else "cpu"
)
print(f"Device      : {device}")
print(f"Checkpoint  : {CKPT_PATH}")
print(f"Output dir  : {OUTPUT_DIR}")

## 2. Cost Matrix

Adjust these to match your business context.

In [ ]:
COST_FN = 1_000.0   # fraud transaction gets through (bank absorbs loss)
COST_FP =   100.0   # legitimate customer is blocked (friction / churn risk)

print(f"Cost of missing fraud (FN) : {COST_FN:>8,.0f} ฿")
print(f"Cost of blocking normal (FP): {COST_FP:>8,.0f} ฿")
print(f"FN/FP cost ratio            : {COST_FN/COST_FP:.1f}×")

## 3. Compute Anomaly Scores

In [ ]:
X_val  = np.load(PROC_DIR / "X_val.npy").astype(np.float32)
y_val  = np.load(PROC_DIR / "y_val.npy").astype(np.int32)
X_test = np.load(PROC_DIR / "X_test.npy").astype(np.float32)
y_test = np.load(PROC_DIR / "y_test.npy").astype(np.int32)

feat_w = _build_feature_weights(device)

scores_val  = compute_anomaly_scores(CKPT_PATH, X_val,  device, feature_weights=feat_w)
scores_test = compute_anomaly_scores(CKPT_PATH, X_test, device, feature_weights=feat_w)

print(f"Val   fraud_rate={y_val.mean()*100:.2f}%  "
      f"score_normal={scores_val[y_val==0].mean():.4f}  "
      f"score_fraud={scores_val[y_val==1].mean():.4f}")
print(f"Test  fraud_rate={y_test.mean()*100:.2f}%  "
      f"score_normal={scores_test[y_test==0].mean():.4f}  "
      f"score_fraud={scores_test[y_test==1].mean():.4f}")

sep_val  = scores_val[y_val==1].mean()  / (scores_val[y_val==0].mean()  + 1e-9)
sep_test = scores_test[y_test==1].mean()/ (scores_test[y_test==0].mean()+ 1e-9)
print(f"Separation  val={sep_val:.2f}×  test={sep_test:.2f}×")

## 4. Threshold Sweep (Validation Set)

Thresholds are selected on the **val set only**.  
Test set is used exclusively for final reporting in Section 6.

In [ ]:
rows_val = threshold_analysis(
    scores_val, y_val, n_points=500, cost_fn=COST_FN, cost_fp=COST_FP
)
opt_val = find_optimal_thresholds(rows_val, min_recall=0.90)

print(f"Max-F1  threshold = {opt_val['max_f1']:.4f}   F1 = {opt_val['max_f1_score']:.4f}")
print(f"MinCost threshold = {opt_val['min_cost']:.4f}   cost = {opt_val['min_cost_value']:,.0f} ฿")
print(f"R≥90%%  threshold = {opt_val['recall90']:.4f}")

## 5. Threshold Curves

In [ ]:
from IPython.display import Image

fig_path = OUTPUT_DIR / "threshold_curves_val.png"
plot_threshold_curves(
    rows_val, opt_val, scores_val, y_val, fig_path,
    cost_fn=COST_FN, cost_fp=COST_FP,
)
Image(str(fig_path), width=1000)

## 6. Detailed Reports at Each Optimal Threshold

Thresholds found on val are now applied to the **held-out test set**.

In [ ]:
thresholds_to_report = {
    "Max F1"       : opt_val["max_f1"],
    "Min Cost"     : opt_val["min_cost"],
    "Recall >= 90%": opt_val["recall90"],
}

test_results = {}
for name, t in thresholds_to_report.items():
    if t < 0:
        print(f"{name}: no threshold achieves recall >= 90% — skipped")
        continue
    test_results[name] = evaluate_at_threshold(
        scores_test, y_test, t,
        cost_fn=COST_FN, cost_fp=COST_FP,
        label=f"{name} (test)",
    )

## 7. Score Distribution — Val Set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

clip = float(np.percentile(scores_val, 99))
bins = np.linspace(0, clip, 80)

ax = axes[0]
ax.hist(scores_val[y_val==0].clip(max=clip), bins=bins, alpha=0.6,
        density=True, color="steelblue", label=f"Normal  n={int((y_val==0).sum()):,}")
ax.hist(scores_val[y_val==1].clip(max=clip), bins=bins, alpha=0.6,
        density=True, color="crimson",   label=f"Fraud   n={int((y_val==1).sum()):,}")
for name, t, color in [
    ("Max F1",   opt_val["max_f1"],   "darkorange"),
    ("Min Cost", opt_val["min_cost"], "green"),
]:
    ax.axvline(t, color=color, linestyle="--", linewidth=1.5, label=name)
ax.set_xlabel("Anomaly score"); ax.set_ylabel("Density")
ax.set_title("Score Distribution — Val Set")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
for label, s, c in [("Normal", scores_val[y_val==0], "steelblue"),
                    ("Fraud",  scores_val[y_val==1], "crimson")]:
    sorted_s = np.sort(s)
    cdf = np.arange(1, len(sorted_s)+1) / len(sorted_s)
    ax.plot(sorted_s, cdf, color=c, label=label)
for name, t, color in [
    ("Max F1",   opt_val["max_f1"],   "darkorange"),
    ("Min Cost", opt_val["min_cost"], "green"),
]:
    ax.axvline(t, color=color, linestyle="--", linewidth=1.5, label=name)
ax.set_xlabel("Anomaly score"); ax.set_ylabel("Cumulative fraction")
ax.set_title("CDF — Normal vs Fraud")
ax.set_xlim(0, clip); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save Thresholds

In [ ]:
output = {
    "cost_fn"    : COST_FN,
    "cost_fp"    : COST_FP,
    "thresholds" : {
        "max_f1"  : opt_val["max_f1"],
        "min_cost": opt_val["min_cost"],
        "recall90": opt_val["recall90"],
    },
    "test_reports": test_results,
}

out_path = OUTPUT_DIR / "thresholds.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved → {out_path}")
print(json.dumps(output["thresholds"], indent=2))

---
## 9. Feature-wise Reconstruction Error

**Question**: which features make fraudulent transactions hard to reconstruct?

Bar chart shows mean per-feature weighted MSE for Normal vs Fraud, sorted by the gap.
Features at the left dominate the anomaly score — they should align with `FEATURE_WEIGHTS`
in `config.py` (V14, V17, V3, …).

In [ ]:
pfe_test = compute_per_feature_errors(CKPT_PATH, X_test, device, feature_weights=feat_w)
print(f"Per-feature error shape: {pfe_test.shape}  (samples × features)")

mean_normal = pfe_test[y_test == 0].mean(axis=0)
mean_fraud  = pfe_test[y_test == 1].mean(axis=0)
delta       = mean_fraud - mean_normal
order       = np.argsort(delta)[::-1]

print("\nTop-10 features by fraud−normal reconstruction error gap:")
print(f"{'Feature':<10} {'Normal MSE':>12} {'Fraud MSE':>12} {'Gap':>10}")
print("-" * 48)
for i in order[:10]:
    print(f"{FEATURE_COLS[i]:<10} {mean_normal[i]:>12.5f} {mean_fraud[i]:>12.5f} {delta[i]:>10.5f}")

In [ ]:
feat_err_path = OUTPUT_DIR / "feature_reconstruction_error.png"
plot_feature_reconstruction_error(pfe_test, y_test, FEATURE_COLS, feat_err_path)
Image(str(feat_err_path), width=1100)

## 10. Anomaly Score vs Transaction Amount

**Question**: is the model biased toward large-amount transactions?

Criminals often start with *small* test transactions before escalating.
If low-amount fraud points fall below the threshold line, the model has an
amount-bias that attackers could exploit.

In [ ]:
amount_test  = X_test[:, AMOUNT_IDX]
scatter_path = OUTPUT_DIR / "score_vs_amount.png"
plot_score_vs_amount(
    scores_test, y_test, amount_test, scatter_path,
    threshold=opt_val["max_f1"],
    threshold_label="Max F1",
)
Image(str(scatter_path), width=900)

# Bias check: do low-amount frauds score lower?
fraud_mask = y_test == 1
med_amount = np.median(amount_test[fraud_mask])
lo = scores_test[fraud_mask & (amount_test <= med_amount)]
hi = scores_test[fraud_mask & (amount_test >  med_amount)]
print(f"Fraud score — low-amount  median: {np.median(lo):.4f}")
print(f"Fraud score — high-amount median: {np.median(hi):.4f}")
print("(low ≈ high → no amount bias; low << high → bias exists)")

## 11. Latent Space T-SNE

**Question**: does the encoder learn to separate fraud in the latent space?

The VAE was trained **only on normal transactions**. If latent μ vectors cluster
fraud points separately, the encoder inferred fraud-discriminative structure
purely from reconstruction pressure — not from labels.

> Runtime: ~30 s on Colab GPU / ~2 min on CPU for 5 000 points.

In [ ]:
X_all = np.concatenate([X_val,  X_test],  axis=0)
y_all = np.concatenate([y_val,  y_test],  axis=0)

mu_all    = compute_latent_mu(CKPT_PATH, X_all, device)
tsne_path = OUTPUT_DIR / "latent_tsne.png"
print(f"Latent μ shape: {mu_all.shape}")

plot_latent_tsne(mu_all, y_all, tsne_path, n_samples=5_000, random_state=42)
Image(str(tsne_path), width=750)

## 12. Confusion Matrix Heatmap

**Question**: in plain numbers — how many customers are blocked and how many frauds are missed?

One heatmap per operating point applied to the **test set**.
This is the format business stakeholders understand without needing to read a PR curve.

In [ ]:
from IPython.display import display, Image as Img

cm_configs = [
    ("Max F1",      opt_val["max_f1"]),
    ("Min Cost",    opt_val["min_cost"]),
    ("Recall≥90%%", opt_val["recall90"]),
]

for label, t in cm_configs:
    if t < 0:
        print(f"{label}: skipped (no threshold achieves recall ≥ 90%)")
        continue
    safe = label.replace(">=", "ge").replace("%%", "pct").replace(" ", "_").lower()
    cm_path = OUTPUT_DIR / f"confusion_matrix_{safe}.png"
    plot_confusion_matrix_heatmap(
        scores_test, y_test, t, cm_path,
        label=label, cost_fn=COST_FN, cost_fp=COST_FP,
    )
    print(f"\n{'─'*40} {label} {'─'*40}")
    display(Img(str(cm_path), width=600))